# 01 · Why before/after comparisons fail

**Question:** if Texas launched a campaign on 2021-01-04, would comparing Texas before vs after the launch measure the campaign's effect?

**Answer:** no. With a true effect of **zero**, Texas shows a **−3.4%** "lift", and the median state with no ads at all moved **−4.4%** over the same weeks. The change is seasonality, not advertising.

**Data:** GA4 Google Merchandise Store public sample (BigQuery `ga4_obfuscated_sample_ecommerce`), aggregated to sessions per US state per day, 2020-11-01 to 2021-01-31. Lifts are **injected** into real data (semi-synthetic), so the true effect is always known.

**Design:** treatment start 2021-01-04; *before* = the 28 days ending 2021-01-03 (12/07–01/03), *after* = the 28 days from the start (01/04–01/31). No lift is injected here, so any non-zero "lift" is pure error.

In [1]:
import pandas as pd
df = pd.read_csv("data/ga4_state_daily.csv", parse_dates=["date"])
df.head()

,date,state,sessions,users,purchases,revenue
0,2020-11-01,Alabama,6,6,0,0.0
1,2020-11-02,Alabama,21,19,0,0.0
2,2020-11-03,Alabama,17,14,0,0.0
3,2020-11-04,Alabama,19,16,0,0.0
4,2020-11-05,Alabama,12,11,0,0.0


## Step 1 · Naive lift for Texas

Naive lift treats the pre-period as the counterfactual: **after ÷ before − 1**.

In [2]:
START = pd.Timestamp("2021-01-04")
WINDOW = pd.Timedelta(days=28)
tx = df[df.state == "Texas"]
before = tx[(tx.date >= START - WINDOW) & (tx.date < START)].sessions.sum()  # 28 days before launch
after  = tx[(tx.date >= START) & (tx.date < START + WINDOW)].sessions.sum()   # 28 days from launch
print(f"Naive lift for Texas: {after / before - 1:.1%}")

Naive lift for Texas: -3.4%


## Step 2 · The same calculation for every state

None of these states ran ads, so their before/after changes show what happens **without** a campaign.

In [3]:
df_before = df[(df.date >= START - WINDOW) & (df.date < START)]
df_after  = df[(df.date >= START) & (df.date < START + WINDOW)]
print(f"Naive lift for Texas: {df_after[df_after.state == 'Texas'].sessions.sum() / df_before[df_before.state == 'Texas'].sessions.sum() - 1:.1%}")

before_by_state = df_before.groupby("state").sessions.sum()
after_by_state  = df_after.groupby("state").sessions.sum()

lift_by_state = after_by_state / before_by_state - 1          # aligned by state
lift_by_state.describe()

Naive lift for Texas: -3.4%


count    51.000000
mean     -0.051672
std       0.091249
min      -0.315789
25%      -0.091791
50%      -0.044248
75%       0.011602
max       0.141892
Name: sessions, dtype: float64

## Findings

| | Before/after change (true effect = 0) |
|---|---|
| Texas | **−3.4%** |
| Median state | **−4.4%** |
| Middle 50% of states | −9.2% to +1.2% |
| Range | −31.6% to +14.2% (std 9.1%) |

- Texas moved about as much as the typical untreated state. The drop comes from **seasonality**: the before window holds holiday shopping, and traffic falls in January.
- Naive lift credits *every* change between the windows to the campaign, so it can't separate the ad effect from time trends (**time-varying confounding**).
- Single states swing ±9% with no ads at all, larger than a typical ad effect.

## Implications

- A geo test needs a **control group** to estimate what would have happened anyway → [02 · Difference-in-differences](02_difference_in_differences.ipynb).
- One geo is too noisy to read a realistic effect; tests need **several geos** and a power analysis → [05](05_power_analysis_mde.ipynb).